# The lattice Schwinger model on four qubits

**Abror Niyazmetov · PHY 420 · Fall 2026 · Advisor: Dr. Kurt Andresen**

Quantum electrodynamics in one space and one time dimension, put on a four-site lattice
and simulated as a quantum circuit.

This notebook is the walkthrough. It does five things in order:

1. Builds the Hamiltonian and prints it as a list of Pauli strings
2. Runs every verification check live, and prints the residuals
3. Shows the actual quantum circuits
4. Measures Trotter error and fits its scaling exponent
5. Measures particle density from finite shots, with error bars, and finds where
   Trotter error drops below shot noise

Nothing here is asserted. Every physics claim is checked against a known answer and the
residual is printed, so you can see how much room is left rather than a pass/fail flag.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
np.set_printoptions(precision=6, suppress=True)

from schwinger.schwinger import (
    hamiltonian, hamiltonian_terms, spectrum, ground_state,
    bare_vacuum, bare_vacuum_label, charge_sectors, format_terms,
    construction_residual, fermion_construction_residual,
)
from schwinger.observables import (
    particle_density, chiral_condensate, electric_field_profile, evolve,
    total_charge, charge_variance,
)
from schwinger.analytics import (
    strong_coupling_ground_energy, free_fermion_spectrum, gauss_law_residual,
    charge_commutator_residual, hellmann_feynman_residual, cp_symmetry_residual,
)
from schwinger.circuits import (
    hamiltonian_op, bare_vacuum_circuit, trotter_step, trotter_circuit,
    statevector, run_with_shots, particle_density_from_counts, circuit_metrics,
)

N = 4          # staggered lattice sites = qubits
M, W, J = 0.5, 1.0, 1.0   # fermion mass, hopping, electric coupling
print(f"N = {N} sites, Hilbert space dimension {2**N}")

N = 4 sites, Hilbert space dimension 16


## 1. The system

Four staggered lattice sites, open boundaries. Even physics sites hold particles, odd
sites hold antiparticles, and an antiparticle is a *hole*, so the vacuum is the filled
Dirac sea rather than an empty lattice.

The gauge field is gone. In one spatial dimension Gauss's law is a difference equation

$$E_n - E_{n-1} = \rho_n$$

which with open boundaries can be **solved** rather than merely imposed:
$E_n = \sum_{k \le n} \rho_k$. So the electric field on every link is fixed by the
charge to its left, and no qubit is spent on it. Four sites, four qubits.

The price is that the electric energy becomes long-ranged in the spins. That long-range
coupling is where confinement lives.

$$H = \frac{m}{2}\sum_{n=1}^{N} (-1)^n Z_n \;+\; w\sum_{n=1}^{N-1}\left(\sigma^+_n\sigma^-_{n+1} + \text{h.c.}\right) \;+\; J\sum_{n=1}^{N-1} L_n^2, \qquad L_n = \tfrac12\sum_{m \le n}\left(Z_m + (-1)^m\right)$$

This is equation (2) of **Martinez et al., Nature 534, 516 (2016)**, who run exactly
this system at N = 4 on a trapped-ion device. Their published equation is what my
independent derivation was checked against.

## 2. The Hamiltonian, as a list of Pauli strings

This is the form a quantum computer accepts, and the form a referee can check.
Site 1 is the leftmost character.

In [2]:
print(format_terms(hamiltonian_terms(N, M, W, J)))

Pauli string      coefficient
------------------------------
IIII                 2.000000
IIIZ                 0.250000
IIZI                -0.750000
IZII                -0.250000
ZIII                -1.250000
IIXX                 0.500000
IIYY                 0.500000
IXXI                 0.500000
IYYI                 0.500000
IZZI                 0.500000
XXII                 0.500000
YYII                 0.500000
ZIZI                 0.500000
ZZII                 1.000000
------------------------------
14 terms


In [3]:
# Symbolic version: each coefficient in terms of m, w, J
def coeffs(m, w, J):
    return {k: v.real for k, v in hamiltonian_terms(N, m, w, J)}
cm, cw, cJ = coeffs(1,0,0), coeffs(0,1,0), coeffs(0,0,1)
keys = sorted(set(list(cm)+list(cw)+list(cJ)), key=lambda s: (sum(c!="I" for c in s), s))
print(f"{'string':<8} {'coefficient':<24} origin")
print("-"*54)
for k in keys:
    parts, origin = [], []
    for val, sym, name in ((cm.get(k,0), "m", "mass"), (cw.get(k,0), "w", "hopping"), (cJ.get(k,0), "J", "electric")):
        if abs(val) > 1e-12:
            parts.append(f"{val:+g} {sym}"); origin.append(name)
    print(f"{k:<8} {' '.join(parts):<24} {', '.join(origin)}")

string   coefficient              origin
------------------------------------------------------
IIII     +2 J                     electric
IIIZ     +0.5 m                   mass
IIZI     -0.5 m -0.5 J            mass, electric
IZII     +0.5 m -0.5 J            mass, electric
ZIII     -0.5 m -1 J              mass, electric
IIXX     +0.5 w                   hopping
IIYY     +0.5 w                   hopping
IXXI     +0.5 w                   hopping
IYYI     +0.5 w                   hopping
IZZI     +0.5 J                   electric
XXII     +0.5 w                   hopping
YYII     +0.5 w                   hopping
ZIZI     +0.5 J                   electric
ZZII     +1 J                     electric


Two features worth pointing at:

- **Site 4 has no ZZ coupling.** Four sites give three links, and $L_3$ involves sites
  1, 2, 3 only. Direct consequence of open boundaries.
- **`ZZII` has twice the coefficient of `ZIZI` and `IZZI`.** The general rule is
  $(J/2)(N - m')$ for the pair $(m, m')$: it grows linearly with distance from the right
  edge. Sites further apart cost more energy because the flux between them is stored on
  every link in between. **That is confinement, written as a coefficient.**

## 3. Verification

Every number below compares two calculations that were done differently. Residuals, not
pass/fail flags.

In [4]:
probes = [(1.0,1.0,1.0), (0.3,1.0,0.7), (2.0,0.5,1.5)]

print("CONSTRUCTION (three independent routes to the same operator)")
print(f"  Pauli expansion vs direct matrix build   : {max(construction_residual(N,*p) for p in probes):.2e}")
print(f"  fermion form through JW vs spin form     : {max(fermion_construction_residual(N,*p) for p in probes):.2e}   <-- Gate 2")

print()
print("ANALYTIC LIMITS")
sc = max(abs(spectrum(N,m,0.0,j)[0] - strong_coupling_ground_energy(N,m))
         for m in (0.5,1.0,2.0) for j in (0.0,1.0,3.0))
ff = max(np.max(np.abs(spectrum(N,0.0,w,0.0) - free_fermion_spectrum(N,w))) for w in (1.0,0.6))
print(f"  strong coupling  E0 = -mN/2 at w = 0     : {sc:.2e}")
print(f"  free fermions, all 16 levels at m = J = 0: {ff:.2e}")
print(f"  Hellmann-Feynman on the condensate       : {max(hellmann_feynman_residual(N,m,1.0,1.0) for m in (0.5,1.5)):.2e}")

print()
print("SYMMETRIES")
print(f"  Gauss's law is an identity               : {gauss_law_residual(N):.2e}")
print(f"  [H, Q] = 0                               : {max(charge_commutator_residual(N,*p) for p in probes):.2e}")
sec, full = cp_symmetry_residual(N, 0.7, 1.0, 1.3)
print(f"  CP inside the neutral sector             : {sec:.2e}")
print(f"  CP on the full space (must NOT vanish)   : {full:.2e}")

print()
print("QISKIT CROSS-CHECK")
print(f"  SparsePauliOp rebuild vs NumPy           : {np.max(np.abs(hamiltonian_op(N,0.7,1.0,1.3).to_matrix() - hamiltonian(N,0.7,1.0,1.3))):.2e}")

CONSTRUCTION (three independent routes to the same operator)
  Pauli expansion vs direct matrix build   : 8.88e-16
  fermion form through JW vs spin form     : 8.88e-16   <-- Gate 2

ANALYTIC LIMITS
  strong coupling  E0 = -mN/2 at w = 0     : 0.00e+00
  free fermions, all 16 levels at m = J = 0: 8.88e-16
  Hellmann-Feynman on the condensate       : 3.22e-11

SYMMETRIES
  Gauss's law is an identity               : 0.00e+00
  [H, Q] = 0                               : 0.00e+00
  CP inside the neutral sector             : 4.44e-16
  CP on the full space (must NOT vanish)   : 6.50e+00

QISKIT CROSS-CHECK
  SparsePauliOp rebuild vs NumPy           : 0.00e+00


Three of these deserve a sentence.

**Strong coupling.** With $w = 0$ the Hamiltonian is diagonal. The electric term is a sum
of squares so it is at least zero, and vanishes only on the bare vacuum. That same
configuration also minimises the mass term. One state minimises both at once, so
$E_0 = -mN/2$ exactly, for any $J$.

**Hellmann-Feynman.** The mass term is exactly $mN$ times the chiral condensate operator,
so $\langle\bar\psi\psi\rangle = (1/N)\,dE_0/dm$. Computing it from the ground-state
vector and from a finite difference of eigenvalues are completely different calculations.
They agree. That catches a wrong Hamiltonian paired with a matching wrong observable,
which no single check can.

**Gauss's law is an identity, not a constraint.** Because it was solved and substituted
rather than imposed, it holds on *every* state in the Hilbert space. So there is nothing
to measure, and the diagnostic in my proposal has to be total charge instead. This is a
correction to the proposal and I would like to talk about it.

## 4. The circuits

State preparation is trivial: the bare vacuum is a product state, so one layer of X gates
makes it. The *true* ground state is entangled and is what state preparation will have to
work for later.

In [5]:
prep = bare_vacuum_circuit(N)
print(f"bare vacuum = |{bare_vacuum_label(N)}>  (site 1 leftmost; |0> = occupied)")
print(prep.draw(output="text"))
print()
print("matches the exact NumPy state:", np.allclose(statevector(prep), bare_vacuum(N)))

bare vacuum = |0101>  (site 1 leftmost; |0> = occupied)
     ┌───┐
q_0: ┤ X ├
     └───┘
q_1: ─────
     ┌───┐
q_2: ┤ X ├
     └───┘
q_3: ─────
          

matches the exact NumPy state: True


One Trotter step. The Hamiltonian splits into three groups whose terms commute
internally, so each group exponentiates exactly:

- **diagonal** (all Z and ZZ terms) as `RZ` and `RZZ`
- **even bonds** of the hopping term as `RXX` and `RYY`
- **odd bonds** likewise

Adjacent bonds share a site and do not commute, which is why the hopping has to be split
into even and odd. All the Trotter error comes from that splitting.

In [6]:
step = trotter_step(N, M, W, J, dt=0.1, order=2)
print(step.draw(output="text", fold=110))

     ┌───────────┐                       ┌────────────┐┌────────────┐                                        »
q_0: ┤ Rz(0.025) ├───────────────────────┤1           ├┤1           ├────────────────────────────────────────»
     ├───────────┴┐                      │  Rxx(0.05) ││  Ryy(0.05) │              ┌───────────┐┌───────────┐»
q_1: ┤ Rz(-0.075) ├─■──────────■─────────┤0           ├┤0           ├──────────────┤1          ├┤1          ├»
     ├────────────┤ │ZZ(0.05)  │         └────────────┘├────────────┤┌────────────┐│  Rxx(0.1) ││  Ryy(0.1) │»
q_2: ┤ Rz(-0.025) ├─■──────────┼────────────■──────────┤1           ├┤1           ├┤0          ├┤0          ├»
     ├────────────┤            │ZZ(0.05)    │ZZ(0.1)   │  Rxx(0.05) ││  Ryy(0.05) │└───────────┘└───────────┘»
q_3: ┤ Rz(-0.125) ├────────────■────────────■──────────┤0           ├┤0           ├──────────────────────────»
     └────────────┘                                    └────────────┘└────────────┘                          »
«

In [7]:
print(f"{'steps':>7} {'dt':>8} {'depth':>8} {'2q gates':>10}   (transpiled to rz/sx/x/cx)")
for s in (10, 50, 200):
    mtr = circuit_metrics(trotter_circuit(N, M, W, J, 3.0, s, order=2))
    print(f"{s:>7} {3.0/s:>8.4f} {mtr['depth']:>8} {mtr['two_qubit_gates']:>10}")

  steps       dt    depth   2q gates   (transpiled to rz/sx/x/cx)
     10   0.3000      541        320


     50   0.0600     2701       1600


    200   0.0150    10801       6400


That table is the tension at the centre of the thesis. Smaller Trotter steps mean less
discretisation error and **more** gates, so more hardware error. The optimum is a
trade-off, and quantifying it is the point.

## 5. Trotter error, and the slope that tests it

The error should fall as $\delta t$ for a first-order product formula and as
$\delta t^2$ for the symmetric second-order one. **The slope is the test, not the size
of the error.** An implementation can be accurate at one step size and still scale
wrongly, which means it is wrong for a reason that will bite at longer times.

In [8]:
T = 3.0
exact_state = evolve(bare_vacuum(N), hamiltonian(N, M, W, J), np.array([T]))[0]

def phase_align(a, b):
    return a * np.exp(-1j*np.angle(np.vdot(b, a)))

steps_list = np.array([10, 20, 40, 80, 160, 320])
dts = T / steps_list
print(f"{'order':>6} {'fitted slope':>14} {'predicted':>11}")
for order in (1, 2):
    errs = [float(np.max(np.abs(phase_align(statevector(trotter_circuit(N,M,W,J,T,int(s),order)), exact_state) - exact_state)))
            for s in steps_list]
    slope = np.polyfit(np.log(dts), np.log(errs), 1)[0]
    print(f"{order:>6} {slope:>14.4f} {float(order):>11.1f}")

 order   fitted slope   predicted


     1         1.0359         1.0


     2         2.0031         2.0


## 6. Pair production, measured with finite shots

Start in the bare vacuum and evolve. Particles appear.

**Where the energy comes from**, because this is the question I expect: nothing is
injected and no external agent pulls charges apart. The bare vacuum is not an eigenstate
of the full Hamiltonian once $w \ne 0$. Total energy is conserved exactly. What changes
is how it is distributed between electric field and rest mass.

In [9]:
H = hamiltonian(N, M, W, J)
v0 = bare_vacuum(N)
print(f"{'m/w':>5} {'E(bare vac)':>13} {'E0 (true)':>11} {'gap':>9} {'peak density':>14}")
tt = np.linspace(0, 12, 481)
for m in (0.0, 0.5, 1.0, 2.0):
    Hm = hamiltonian(N, m, W, J)
    e_bare = float(np.vdot(v0, Hm @ v0).real)
    e0, _ = ground_state(N, m, W, J)
    peak = max(particle_density(s, N) for s in evolve(v0, Hm, tt))
    print(f"{m:>5} {e_bare:>13.6f} {e0:>11.6f} {e_bare-e0:>9.6f} {peak:>14.6f}")
print()
print("The gap column is the whole story: a larger mismatch between the prepared state")
print("and the state the Hamiltonian wants gives more pair production.")

  m/w   E(bare vac)   E0 (true)       gap   peak density
  0.0      0.000000   -1.477352  1.477352       0.612095


  0.5     -1.000000   -2.075478  1.075478       0.462907
  1.0     -2.000000   -2.828162  0.828162       0.334990


  2.0     -4.000000   -4.553129  0.553129       0.175162

The gap column is the whole story: a larger mismatch between the prepared state
and the state the Hamiltonian wants gives more pair production.


Now measure it the way a real device would: run the circuit a finite number of times
and count.

The particle-number operator $N_{\text{part}} = \tfrac12\sum_n [I + (-1)^n Z_n]$ is
**diagonal in Z**, so one measurement basis gives it. No basis rotations, no separate
circuit per term. Most variational observables do not have that property.

In [10]:
t = 3.0
exact_density = particle_density(exact_state, N)
qc = trotter_circuit(N, M, W, J, t, 200, order=2)

print(f"exact particle density at t = {t}: {exact_density:.6f}")
print()
print(f"{'shots':>7} {'estimate':>11} {'uncertainty':>13} {'deviation':>11} {'sigmas':>8}")
for shots in (128, 512, 2048, 8192, 32768):
    est, err = particle_density_from_counts(run_with_shots(qc, shots, seed=20260910), N)
    print(f"{shots:>7} {est:>11.5f} {err:>13.5f} {est-exact_density:>11.5f} {abs(est-exact_density)/err:>8.2f}")

exact particle density at t = 3.0: 0.291507

  shots    estimate   uncertainty   deviation   sigmas


    128     0.26562       0.02528    -0.02588     1.02


    512     0.30078       0.01313     0.00927     0.71


   2048     0.29053       0.00664    -0.00098     0.15


   8192     0.28949       0.00335    -0.00202     0.60


  32768     0.29002       0.00167    -0.00148     0.89


Uncertainty falls as $1/\sqrt{\text{shots}}$, and the deviation stays within a couple
of sigma. That is shot noise behaving.

## 7. The crossover

At what Trotter step does discretisation error drop below shot noise? Below that step
size, shrinking $\delta t$ buys nothing: you are paying in circuit depth for an
improvement the statistics cannot see.

This is the first of the two headline results in my proposal. Gate error is not in it
yet, which is what turns it from a two-way into a three-way trade.

In [11]:
steps_c = np.array([2, 4, 8, 16, 32, 64, 128, 256])
sysm = np.array([abs(particle_density(statevector(trotter_circuit(N,M,W,J,T,int(s),2)), N) - exact_density)
                 for s in steps_c])

for shots in (256, 4096):
    _, sigma = particle_density_from_counts(run_with_shots(trotter_circuit(N,M,W,J,T,200,2), shots, seed=20260910), N)
    below = T/steps_c[sysm < sigma]
    print(f"{shots:>5} shots: shot noise sigma = {sigma:.5f}  ->  Trotter error falls below it at dt ~ {below.max():.4f}")

print()
print(f"{'dt':>8} {'Trotter error':>15} {'2q gates':>10}")
for s, e in zip(steps_c, sysm):
    print(f"{T/s:>8.4f} {e:>15.2e} {circuit_metrics(trotter_circuit(N,M,W,J,T,int(s),2))['two_qubit_gates']:>10}")

  256 shots: shot noise sigma = 0.01760  ->  Trotter error falls below it at dt ~ 0.1875


 4096 shots: shot noise sigma = 0.00471  ->  Trotter error falls below it at dt ~ 0.0938

      dt   Trotter error   2q gates
  1.5000        1.67e-01         64
  0.7500        1.94e-01        128
  0.3750        4.29e-02        256
  0.1875        1.02e-02        512
  0.0938        2.51e-03       1024
  0.0469        6.26e-04       2048
  0.0234        1.56e-04       4096


  0.0117        3.91e-05       8192


## 8. Where this stands

**Done.** The Hamiltonian is derived, built three independent ways, and checked against
six analytic limits. State preparation and Trotterised evolution work as circuits, and
the Trotter exponent is measured rather than assumed. Shot noise is in.

**Not done.** Gate and readout error, which is the other half of the error budget.
Ground-state preparation on the device. The full separation of Trotter, statistical, and
hardware error onto each observable, which is the actual thesis.

**Two things I would like to discuss.**

1. The Gauss's law diagnostic in my proposal does not exist in this encoding, because the
   constraint is substituted rather than imposed. I have replaced it with two others,
   total charge variance and a CP symmetry, both exact and both broken by noise. I think
   this is a strengthening but I want it checked.
2. The measurement side is where I most need supervision: whether my uncertainties are
   propagated correctly, and whether I am claiming more than the data supports.